# Entrega 3 — Chunking de Manifestações Longas
**Objetivo:** Dividir textos grandes (> 500 caracteres) em pedaços (chunks) preservando contexto, para evitar perda semântica em indexações únicas.


In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA

# 1. Filtrar manifestações longas
with open('manifestacoes.json', 'r', encoding='utf-8') as f:
    df = pd.DataFrame(json.load(f))

df['tamanho'] = df['texto'].apply(len)
longas = df[df['tamanho'] > 500].copy()
print(f"Encontradas {len(longas)} manifestações longas.")

modelo = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")


In [ ]:
# 2. Testando Configurações de Chunking
texto_teste = longas.iloc[0]['texto']
print(f"Texto Original (M011): {texto_teste[:150]}...\n")

# Config 1: Baixo overlap
splitter_1 = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)
chunks_1 = splitter_1.split_text(texto_teste)

# Config 2: Alto overlap
splitter_2 = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=100)
chunks_2 = splitter_2.split_text(texto_teste)

print(f"Config 1 gerou {len(chunks_1)} chunks. (Overlap: 20)")
print("Chunk 1:", chunks_1[0])
print("Chunk 2:", chunks_1[1])
print("\n" + "="*50 + "\n")
print(f"Config 2 gerou {len(chunks_2)} chunks. (Overlap: 100)")
print("Chunk 1:", chunks_2[0])
print("Chunk 2:", chunks_2[1])


### Respostas:
1. **Como o overlap influencia a coesão semântica?** O overlap (sobreposição) carrega o final do contexto do chunk anterior para o início do próximo. Com baixo overlap (20), pronomes e sujeitos podem ser perdidos. Com overlap alto (100), o Chunk 2 herda a menção ao "buraco e à Avenida Principal", garantindo que um modelo de busca entenda do que se trata aquele pedaço isolado.
2. **Qual configuração preserva melhor?** A configuração 2 (Overlap 100) preserva melhor o sentido isolado das denúncias. Por exemplo, se a denúncia longa mudar de assunto (de buraco para mato alto), o overlap garante que a transição não corte uma frase ao meio de forma abrupta, permitindo que a busca vetorial encontre o fragmento exato sobre "mato".


In [ ]:
# 3. Visualização 2D (PCA) de Chunks de Múltiplos Textos
chunks_totais = []
labels = []

# Usaremos a Config 2 para todos os textos longos
for idx, row in longas.iterrows():
    pedacos = splitter_2.split_text(row['texto'])
    chunks_totais.extend(pedacos)
    labels.extend([row['id']] * len(pedacos))

embeddings_chunks = modelo.encode(chunks_totais)
pca = PCA(n_components=2)
coords = pca.fit_transform(embeddings_chunks)

plt.figure(figsize=(10, 7))
unique_labels = list(set(labels))
cores = plt.cm.tab10(range(len(unique_labels)))

for id_doc, cor in zip(unique_labels, cores):
    idx_mask = [i for i, lbl in enumerate(labels) if lbl == id_doc]
    plt.scatter(coords[idx_mask, 0], coords[idx_mask, 1], label=id_doc, s=100, color=cor)
    # Ligando os chunks do mesmo documento
    plt.plot(coords[idx_mask, 0], coords[idx_mask, 1], color=cor, alpha=0.3, linestyle='--')

plt.title("PCA dos Chunks das Manifestações Longas")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


**Análise Visual:** Sim, os chunks de uma mesma manifestação tendem a ficar agrupados (clusters naturais) e a linha tracejada mostra a trajetória narrativa. Isso ocorre porque o vocabulário e o contexto semântico se mantêm consistentes ao longo do relato do cidadão.
